In [1]:
import pandas as pd
import json

file_path = './processed_videos_2.json'

print(f'carregando {file_path}')
data = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except e:
            pass

df = pd.DataFrame(data)
initial_count = len(df)
df = df.drop_duplicates(subset=['video_id'], keep='first')
removed_count = initial_count - len(df)
print(f'{removed_count} duplicatas')
print(f'{len(df)} videos')

carregando ./processed_videos_2.json
3 duplicatas
21183 videos


In [3]:
from openai import OpenAI
import time
import os
from tqdm import tqdm

print(f'carregando {file_path}')
data = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except e:
            pass

df = pd.DataFrame(data)
initial_count = len(df)
df = df.drop_duplicates(subset=['video_id'], keep='first').reset_index(drop=True)
removed_count = initial_count - len(df)
print(f'{removed_count} duplicatas')
print(f'{len(df)} videos')


client = OpenAI()

OUTPUT_JSON = "resultados_vacinacao_2.jsonl"
OUTPUT_LOG = 'model_input_2.txt'
BATCH_SIZE = 1                        # Quantos vídeos por chamada
SLEEP_TIME = 1.0                       # Intervalo entre chamadas (em segundos)
MODEL = "gpt-4o-mini"

PROMPT_BASE = """Você é um assistente especializado em análise de conteúdo sobre saúde pública.

Tarefa: Leia o título e a descrição de cada vídeo e determine se ele é relevante para o contexto de SAÚDE PÚBLICA e, mais especificamente, para discussões sobre VACINAÇÃO (imunização, campanhas de vacinação, vacinas contra doenças, hesitação vacinal, efeitos adversos, políticas públicas de saúde, desinformação, fake news, etc.).

Critérios:

Responda "Sim" se o vídeo claramente:
- Trata de vacinas, vacinação, imunização ou campanhas de saúde pública;
- Discute doenças preveníveis por vacina (como COVID-19, gripe, sarampo, HPV, etc.);
- Fala sobre segurança, eficácia ou obrigatoriedade de vacinas;
- Menciona desinformação e/ou teorias da conspiração sobre vacinas;
- Apresenta depoimentos, críticas ou debates sobre o uso de vacinas em humanos.

Responda "Não" se o vídeo:
- Usa “vacina” fora do contexto de saúde (ex: "vacina contra o tédio");
- Trata de ficção, metáforas, humor ou religião sem relação clara com vacinação real;
- Cita “vacina” apenas de passagem, sem relação relevante com saúde pública.

Responda **apenas** com uma lista numerada, com exatamente o mesmo número de respostas que vídeos apresentados, e na mesma ordem:

Agora avalie os seguintes vídeos:
"""



done_ids = set()
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                record = json.loads(line)
                done_ids.add(record['id'])
            except Exception:
                pass

for start in tqdm(range(0, len(df), BATCH_SIZE)):
    end = start + BATCH_SIZE
    batch = df.iloc[start:end]

    if all(row['video_id'] in done_ids for _, row in batch.iterrows()):
        continue

    videos_text = ""
    for i, row in batch.iterrows():
        title = str(row['title']).strip()
        desc = str(row.get('description', '')).strip()
        videos_text += f"\n\n Vídeo {i+1}:\nTítulo: {title}\nDescrição: {desc}"

    full_prompt = PROMPT_BASE + videos_text
    #print(full_prompt[:1000])

    
    try:
        
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{'role':'user', 'content':full_prompt}],
            temperature=0,
        )
        text = response.choices[0].message.content.strip().lower()
        
        with open(OUTPUT_LOG, 'a', encoding='utf-8') as f_log:
            f_log.write(f'\n\n-- Lote {start}:{end} --\n')
            f_log.write(full_prompt)
            f_log.write('\n--------')

        
        replies = []        
        for line in text.split('\n'):
            line = line.strip().lower()
            if 'sim' in line:
                replies.append('sim')
            elif 'não' in line or 'nao' in line:
                replies.append('não')
            else:
                print(f'saída: {line}')
        
        #print(replies)
        results = []
        for (_, row), resp in zip(batch.iterrows(), replies):
            video_id = row['video_id']
            if video_id in done_ids:
                continue
            
            res = {
                'id' : video_id,
                'classification' : resp,
                'title' : row['title'],
                'description' : row['description']
            }
            results.append(res)
            done_ids.add(video_id)
        
        with open(OUTPUT_JSON, 'a', encoding='utf-8') as f:
            for r in results:
                json.dump(r, f, ensure_ascii=False)
                f.write('\n')
        
        #time.sleep(1)
        
    except Exception as e:
        print(f'erro {e}')
        #time.sleep(SLEEP_TIME)
        

carregando ./processed_videos_2.json
3 duplicatas
21183 videos


 12%|█████████                                                                  | 2573/21183 [22:41<2:44:09,  1.89it/s]


KeyboardInterrupt: 